# Lab | Error Handling

## Exercise: Error Handling for Managing Customer Orders

The implementation of your code for managing customer orders assumes that the user will always enter a valid input. 

For example, we could modify the `initialize_inventory` function to include error handling.
   - If the user enters an invalid quantity (e.g., a negative value or a non-numeric value), display an error message and ask them to re-enter the quantity for that product.
   - Use a try-except block to handle the error and continue prompting the user until a valid quantity is entered.

```python
# Step 1: Define the function for initializing the inventory with error handling
def initialize_inventory(products):
    inventory = {}
    for product in products:
        valid_quantity = False
        while not valid_quantity:
            try:
                quantity = int(input(f"Enter the quantity of {product}s available: "))
                if quantity < 0:
                    raise ValueError("Invalid quantity! Please enter a non-negative value.")
                valid_quantity = True
            except ValueError as error:
                print(f"Error: {error}")
        inventory[product] = quantity
    return inventory

# Or, in another way:

def initialize_inventory(products):
    inventory = {}
    for product in products:
        valid_input = False
        while not valid_input:
            try:
                quantity = int(input(f"Enter the quantity of {product}s available: "))
                if quantity >= 0:
                    inventory[product] = quantity
                    valid_input = True
                else:
                    print("Quantity cannot be negative. Please enter a valid quantity.")
            except ValueError:
                print("Invalid input. Please enter a valid quantity.")
    return inventory
```

Let's enhance your code by implementing error handling to handle invalid inputs.

Follow the steps below to complete the exercise:

2. Modify the `calculate_total_price` function to include error handling.
   - If the user enters an invalid price (e.g., a negative value or a non-numeric value), display an error message and ask them to re-enter the price for that product.
   - Use a try-except block to handle the error and continue prompting the user until a valid price is entered.

3. Modify the `get_customer_orders` function to include error handling.
   - If the user enters an invalid number of orders (e.g., a negative value or a non-numeric value), display an error message and ask them to re-enter the number of orders.
   - If the user enters an invalid product name (e.g., a product name that is not in the inventory), or that doesn't have stock available, display an error message and ask them to re-enter the product name. *Hint: you will need to pass inventory as a parameter*
   - Use a try-except block to handle the error and continue prompting the user until a valid product name is entered.

4. Test your code by running the program and deliberately entering invalid quantities and product names. Make sure the error handling mechanism works as expected.


## Solution
The customer order remains a set, as requested in the preceding Functions lab. Duplicate products are rejected and the requested count cannot exceed the number of distinct products in stock. Inventory is updated after collecting orders. Prices reject negative, non-numeric and non-finite values.

In [1]:
from math import isfinite

def initialize_inventory(products):
    inventory = {}
    for product in products:
        while True:
            try:
                quantity = int(input(f"Enter the quantity of {product}s available: "))
                if quantity < 0:
                    raise ValueError("Quantity must be non-negative.")
                inventory[product] = quantity
                break
            except ValueError as error:
                print(f"Error: {error}")
    return inventory

def get_customer_orders(inventory):
    # The Functions lab uses a set: each product can be ordered once.
    available = {product for product, quantity in inventory.items() if quantity > 0}
    while True:
        try:
            count = int(input("Enter the number of unique products to order: "))
            if count < 0:
                raise ValueError("Number of orders must be non-negative.")
            if count > len(available):
                raise ValueError(f"Only {len(available)} distinct products are in stock.")
            break
        except ValueError as error:
            print(f"Error: {error}")
    customer_orders = set()
    while len(customer_orders) < count:
        try:
            product = input("Enter a product name: ").strip()
            if product not in inventory:
                raise ValueError("Product is not in the inventory.")
            if inventory[product] <= 0:
                raise ValueError("Product is out of stock.")
            if product in customer_orders:
                raise ValueError("This product is already in the order.")
            customer_orders.add(product)
        except ValueError as error:
            print(f"Error: {error}")
    return customer_orders

def update_inventory(customer_orders, inventory):
    for product in customer_orders:
        if product not in inventory or inventory[product] <= 0:
            raise ValueError(f"Product unavailable: {product}")
    for product in customer_orders:
        inventory[product] -= 1

def calculate_order_statistics(customer_orders, products):
    total = len(customer_orders)
    percentage = 100 * total / len(products) if products else 0
    return total, percentage

def print_order_statistics(order_statistics):
    total, percentage = order_statistics
    print(f"Total products ordered: {total}")
    print(f"Percentage of unique products ordered: {percentage:.2f}%")

def print_updated_inventory(inventory):
    for product, quantity in inventory.items():
        print(f"{product}: {quantity}")

def calculate_total_price(customer_orders):
    total = 0.0
    for product in sorted(customer_orders):
        while True:
            try:
                price = float(input(f"Enter the price of {product}: "))
                if not isfinite(price) or price < 0:
                    raise ValueError("Price must be a finite non-negative number.")
                total += price
                break
            except ValueError as error:
                print(f"Error: {error}")
    return total

def run_program(products):
    inventory = initialize_inventory(products)
    customer_orders = get_customer_orders(inventory)
    total_price = calculate_total_price(customer_orders)
    update_inventory(customer_orders, inventory)
    print_order_statistics(calculate_order_statistics(customer_orders, products))
    print_updated_inventory(inventory)
    print(f"Total price: {total_price:.2f}")
    return inventory, customer_orders, total_price


## Tests with deliberately invalid input
Mocked input makes the interactive retry paths reproducible without manual typing.

In [2]:
from unittest.mock import patch
from contextlib import redirect_stdout
from io import StringIO

def simulate(function, answers, *args):
    output = StringIO()
    with patch("builtins.input", side_effect=answers), redirect_stdout(output):
        result = function(*args)
    return result, output.getvalue()

inventory, log = simulate(initialize_inventory, ["bad", "-1", "2", "0"], ["apple", "pear"])
assert inventory == {"apple": 2, "pear": 0}
assert log.count("Error:") == 2
orders, log = simulate(get_customer_orders, ["bad", "-1", "2", "1", "banana", "pear", "apple"], inventory)
assert orders == {"apple"}
assert log.count("Error:") == 5
orders2, log = simulate(get_customer_orders, ["2", "apple", "apple", "pear"], {"apple": 2, "pear": 1})
assert orders2 == {"apple", "pear"} and log.count("Error:") == 1
price, log = simulate(calculate_total_price, ["bad", "-1", "nan", "inf", "2.5"], orders)
assert price == 2.5 and log.count("Error:") == 4
assert simulate(get_customer_orders, ["1", "0"], {"apple": 0})[0] == set()
assert calculate_total_price(set()) == 0
assert calculate_order_statistics(set(), []) == (0, 0)
assert calculate_order_statistics({"apple"}, ["apple", "pear"]) == (1, 50)
update_inventory(orders, inventory)
assert inventory == {"apple": 1, "pear": 0}
try:
    update_inventory({"pear"}, inventory)
except ValueError:
    pass
else:
    raise AssertionError("Out-of-stock update should fail")
result, log = simulate(run_program, ["bad", "-1", "2", "0", "-1", "1", "banana", "pear", "apple", "bad", "-2", "3.5"], ["apple", "pear"])
assert result == ({"apple": 1, "pear": 0}, {"apple"}, 3.5)
assert "Total price: 3.50" in log
print("All lab checks passed, including the complete interactive program.")
print(log)


All lab checks passed, including the complete interactive program.
Error: invalid literal for int() with base 10: 'bad'
Error: Quantity must be non-negative.
Error: Number of orders must be non-negative.
Error: Product is not in the inventory.
Error: Product is out of stock.
Error: could not convert string to float: 'bad'
Error: Price must be a finite non-negative number.
Total products ordered: 1
Percentage of unique products ordered: 50.00%
apple: 1
pear: 0
Total price: 3.50



## Run interactively
Uncomment the next line and choose your product names to run the program.

In [ ]:
# run_program(["apple", "pear", "banana"])